In [1]:
"""
pctc_plots.py
=============
Publication-quality figures for PCTC m=1 benchmarked on H2-1E.

Loads:  pctc_metrics.json  (produced by pctc_nexus_benchmark.py)

If compiled metrics (N_2q_compiled etc.) are None/null, the script renders
"PENDING" placeholders and still shows logical metrics + noise box.
Once you have real compiled data, re-run this script — no changes needed.

Saves:
  fig_pctc_bar_1x4.pdf/png    — 4-panel bar (one per metric)
  fig_pctc_bar_2x2.pdf/png    — 2x2 layout
  fig_pctc_summary.pdf/png    — single grouped bar, best for paper

Fidelity model (compiled circuit only):
    F = (1 - p2)^N_2q  x  (1 - p_ro)^N_meas
"""

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

In [2]:
# Load PCTC metrics
with open("/Users/nandan/Desktop/CTCs/pctc_metrics.json") as f:
    d = json.load(f)

compiled_available = d.get("N_2q_compiled") is not None

# H2-1E noise specs (calibration 2025-04-30)
NOISE = {
    "date":   "2025-04-30",
    "p1":     1.89e-5,
    "p1_unc": 4.23e-6,
    "p2":     1.05e-3,
    "p2_unc": 8.08e-5,
    "p_ro":   1.39e-3,   # p(0|1) dominant readout error
    "p_ro_0": 6.00e-4,   # p(1|0)
    "p_mem":  2.03e-4,
}

# Logical metrics (always available)
n2q_log   = d["N_2q_logical"]
n1q_log   = d["N_1q_logical"]
d_log     = d["d_logical"]
ntot_log  = d["N_gates_total_logical"]
n_meas    = d["N_meas"]

# Compiled metrics (may be None)
n2q_comp  = d.get("N_2q_compiled")
n1q_comp  = d.get("N_1q_compiled")
d_comp    = d.get("d_compiled")
ntot_comp = d.get("N_gates_total")

In [5]:
# Fidelity
if compiled_available:
    F = (1 - NOISE["p2"])**n2q_comp * (1 - NOISE["p_ro"])**n_meas
    F_str = f"$F = {F:.4f}$"
    print(f"PCTC m=1  N_2q={n2q_comp}  N_meas={n_meas}  ->  F = {F:.4f}")
else:
    F = None
    F_str = r"$F$ = pending compiled data"
    print("Compiled metrics not yet available -- rendering logical metrics only.")

# Style
plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      11,
    "axes.linewidth": 0.8,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "xtick.top":      True,
    "ytick.right":    True,
})

COLORS        = {"log": "#4c72b0", "comp": "#dd8452"}
FID_COLOR     = "#2ca02c"
PENDING_COLOR = "#aaaaaa"
OUT = "/Users/nandan/Desktop/CTCs/Figures"

NOISE_BOX_TXT = (
    "H2-1E noise ({})\n"
    "$p_1 = {:.2e}$\n"
    "$p_2 = {:.2e}$\n"
    "$p_{{\\rm ro}} = {:.2e}$"
).format(NOISE["date"], NOISE["p1"], NOISE["p2"], NOISE["p_ro"])

PCTC m=1  N_2q=12  N_meas=3  ->  F = 0.9834


In [15]:
# (sym, log_val, comp_val_or_None, ylabel)
METRICS = [
    (r"$N_{2q}$",             n2q_log,  n2q_comp,  r"Two-qubit gate count  $N_{2q}$"),
    (r"$d$",                  d_log,    d_comp,    r"Total circuit depth  $d$"),
    (r"$N_{1q}$",             n1q_log,  n1q_comp,  r"Single-qubit gate count  $N_{1q}$"),
    (r"$N_{\mathrm{gates}}$", ntot_log, ntot_comp, r"Total gate count  $N_{\mathrm{gates}}$"),
]


def _bar_panel(ax, sym, log_val, comp_val, ylabel,
               show_noise=False, show_fidelity=False):
    if comp_val is not None:
        labels = ["Logical", "Compiled\n(H2-1E)"]
        vals   = [log_val, comp_val]
        colors = [COLORS["log"], COLORS["comp"]]
    else:
        labels = ["Logical", "Compiled\n(pending)"]
        vals   = [log_val, log_val * 0.01]
        colors = [COLORS["log"], PENDING_COLOR]

    x    = np.arange(2)
    w    = 0.45
    bars = ax.bar(x, vals, w, color=colors, alpha=0.90,
                  edgecolor="white", linewidth=0.8)
    ymax = max(log_val, comp_val if comp_val is not None else log_val)

    for bar, v, col, lbl in zip(bars, vals, colors, labels):
        if "pending" in lbl:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    ymax * 0.08, "?",
                    ha="center", va="bottom",
                    fontsize=14, color=PENDING_COLOR, fontweight="bold")
        else:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + ymax * 0.03, str(int(v)),
                    ha="center", va="bottom",
                    fontsize=12, color=col, fontweight="bold")

    if comp_val is not None and comp_val != log_val:
        ratio = comp_val / log_val
        sign  = "up" if comp_val > log_val else "down"
        arrow = "\u2191" if comp_val > log_val else "\u2193"
        ax.text(0.5, 0.91, f"{arrow} x{ratio:.1f}",
                transform=ax.transAxes, ha="center",
                fontsize=10, color="#555555", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10.5)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(0, ymax * 1.28)
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))

    if show_fidelity:
        ax.text(0.97, 0.97, F_str,
                transform=ax.transAxes, ha="right", va="top",
                fontsize=10, color=FID_COLOR, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="honeydew",
                          edgecolor=FID_COLOR, alpha=0.85))

    if show_noise:
        ax.text(0.97, 0.04, NOISE_BOX_TXT,
                transform=ax.transAxes, fontsize=8,
                verticalalignment="bottom", horizontalalignment="right",
                fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow",
                          edgecolor="#aaaaaa", alpha=0.85))


def _shared_legend(fig):
    handles = [
        mpatches.Patch(color=COLORS["log"],  label="Logical"),
        mpatches.Patch(color=COLORS["comp"] if compiled_available else PENDING_COLOR,
                       label="Compiled (H2-1E)" if compiled_available else "Compiled (pending)"),
    ]
    if F is not None:
        handles.append(
            plt.Line2D([0], [0], linestyle="none", marker="$F$",
                       color=FID_COLOR, markersize=10,
                       label=r"$F=(1-p_2)^{N_{2q}}(1-p_{\rm ro})^{N_{\rm meas}}$")
        )
    fig.legend(handles=handles, loc="lower center", ncol=3,
               fontsize=10, frameon=True,
               framealpha=0.9, edgecolor="#cccccc",
               bbox_to_anchor=(0.5, -0.06))


def fig_bar_1x4():
    fig, axes = plt.subplots(1, 4, figsize=(17, 5.0))
    fig.suptitle(
        r"PCTC $m=1$: Logical vs Compiled (H2-1E) -- Resource Overhead",
        fontsize=13, fontweight="bold", y=1.03
    )
    for i, (ax, (sym, lv, cv, ylabel)) in enumerate(zip(axes, METRICS)):
        _bar_panel(ax, sym, lv, cv, ylabel,
                   show_noise=(i == 0), show_fidelity=(i == 0))
        ax.set_title(sym, fontsize=13, pad=6, fontweight="bold")
    _shared_legend(fig)
    plt.tight_layout()
    fig.savefig(f"{OUT}/fig_pctc_bar_1x4.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(f"{OUT}/fig_pctc_bar_1x4.png", bbox_inches="tight", dpi=300)
    print("Saved fig_pctc_bar_1x4")
    plt.close()


def fig_bar_2x2():
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    fig.suptitle(
        r"PCTC $m=1$: Logical vs Compiled (H2-1E)" + "\nResource Overhead",
        fontsize=13, fontweight="bold", y=1.02
    )
    for i, (ax, (sym, lv, cv, ylabel)) in enumerate(zip(axes.flat, METRICS)):
        _bar_panel(ax, sym, lv, cv, ylabel,
                   show_noise=(i == 0), show_fidelity=(i == 0))
        ax.set_title(sym, fontsize=13, pad=6, fontweight="bold")
    _shared_legend(fig)
    plt.tight_layout()
    fig.savefig(f"{OUT}/fig_pctc_bar_2x2.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(f"{OUT}/fig_pctc_bar_2x2.png", bbox_inches="tight", dpi=300)
    print("Saved fig_pctc_bar_2x2")
    plt.close()


def fig_summary():
    fig, ax = plt.subplots(figsize=(10, 5.5))
    fid_title = f"  |  $F = {F:.4f}$" if F is not None else ""
    fig.suptitle(
        r"PCTC $m=1$ on H2-1E: Logical vs Compiled Resource Counts" + fid_title,
        fontsize=13, fontweight="bold", y=1.02
    )

    metric_labels = [r"$N_{2q}$", r"$d$", r"$N_{1q}$", r"$N_{\rm gates}$"]
    log_vals  = [n2q_log,  d_log,  n1q_log,  ntot_log]
    comp_vals = [n2q_comp, d_comp, n1q_comp, ntot_comp]

    x = np.arange(len(metric_labels))
    w = 0.35

    b1 = ax.bar(x - w/2, log_vals, w,
                color=COLORS["log"], alpha=0.90,
                edgecolor="white", linewidth=0.7, label="Logical")

    if compiled_available:
        b2 = ax.bar(x + w/2, comp_vals, w,
                    color=COLORS["comp"], alpha=0.90,
                    edgecolor="white", linewidth=0.7, label="Compiled (H2-1E)")
    else:
        b2 = ax.bar(x + w/2, [v * 0.01 for v in log_vals], w,
                    color=PENDING_COLOR, alpha=0.60,
                    edgecolor="white", linewidth=0.7, label="Compiled (pending)")

    ymax = max(log_vals)

    for rect in b1:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width() / 2,
                h + ymax * 0.02, str(int(h)),
                ha="center", va="bottom",
                fontsize=10, color=COLORS["log"], fontweight="bold")

    for i, rect in enumerate(b2):
        if compiled_available:
            h = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2,
                    h + ymax * 0.02, str(int(h)),
                    ha="center", va="bottom",
                    fontsize=10, color=COLORS["comp"], fontweight="bold")
        else:
            ax.text(rect.get_x() + rect.get_width() / 2,
                    ymax * 0.06, "?",
                    ha="center", va="bottom",
                    fontsize=14, color=PENDING_COLOR, fontweight="bold")

    if compiled_available:
        for i, (lv, cv) in enumerate(zip(log_vals, comp_vals)):
            ratio = cv / lv
            arrow = "\u2191" if cv > lv else "\u2193"
            ax.text(i, max(lv, cv) + ymax * 0.09,
                    f"{arrow}x{ratio:.1f}",
                    ha="center", fontsize=9.5, color="#555555", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=13)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_ylim(0, 45)
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))

    # Noise box bottom-left (avoids overlap with bar value labels)
    ax.text(0.02, 0.98, NOISE_BOX_TXT,
            transform=ax.transAxes, fontsize=9,
            verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="lightyellow",
                      edgecolor="#aaaaaa", alpha=0.90))

    ax.legend(fontsize=11, framealpha=0.9, edgecolor="#cccccc", loc="upper center")
    plt.tight_layout()
    fig.savefig(f"{OUT}/fig_pctc_summary.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(f"{OUT}/fig_pctc_summary.png", bbox_inches="tight", dpi=300)
    print("Saved fig_pctc_summary")
    plt.close()


def _scatter_panel(ax, log_val, comp_val, sym, title, show_noise=False):
    # Axis range with padding
    lo = min(log_val, comp_val) * 0.60
    hi = max(log_val, comp_val) * 1.45
    ax.plot([lo, hi], [lo, hi], "--", color="black", lw=0.9,
            alpha=0.35, zorder=1, label="$y = x$")

    # Single data point
    ax.scatter(log_val, comp_val, s=120, color="black",
               marker="o", zorder=4)
    ax.annotate("$m=1$", (log_val, comp_val),
                textcoords="offset points", xytext=(8, 5),
                fontsize=10, color="black")

    # Fidelity label (only meaningful for N_2q panel)
    if compiled_available and F is not None and "N_{2q}" in sym:
        ax.annotate(f"$F={F:.4f}$", (log_val, comp_val),
                    textcoords="offset points", xytext=(8, -14),
                    fontsize=9, color=FID_COLOR, style="italic")

    # Overhead annotation
    if comp_val != log_val:
        ratio = comp_val / log_val
        arrow = "\u2191" if comp_val > log_val else "\u2193"
        ax.text(0.05, 0.92, f"{arrow} \u00d7{ratio:.1f}",
                transform=ax.transAxes, fontsize=10,
                color="#555555", style="italic")

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f"Logical  {sym}", fontsize=11)
    ax.set_ylabel(f"Compiled  {sym}", fontsize=11)
    ax.set_title(title, fontsize=10.5, pad=7, fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.45, color="grey")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_aspect("equal", adjustable="box")

    if show_noise:
        ax.text(0.03, 0.60, NOISE_BOX_TXT,
                transform=ax.transAxes,
                fontsize=8, verticalalignment="top",
                fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow",
                          edgecolor="#aaaaaa", alpha=0.85))


def _scatter_legend(fig):
    handles = [
        plt.Line2D([0], [0], linestyle="--", color="black",
                   alpha=0.4, lw=1.2, label="$y = x$  (no overhead)"),
        plt.scatter([], [], s=80, color="black", marker="o", label="$m=1$"),
    ]
    if F is not None:
        handles.append(
            plt.Line2D([0], [0], linestyle="none", marker="$F$",
                       color=FID_COLOR, markersize=10,
                       label=r"$F=(1-p_2)^{N_{2q}}(1-p_{\rm ro})^{N_{\rm meas}}$")
        )
    fig.legend(handles=handles, loc="lower center", ncol=3,
               fontsize=10, frameon=True,
               framealpha=0.9, edgecolor="#cccccc",
               bbox_to_anchor=(0.5, -0.08))


def fig_scatter_1x4():
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
    fig.suptitle(
        r"PCTC $m=1$: Logical vs Compiled (H2-1E) — Resource Overhead",
        fontsize=13, fontweight="bold", y=1.04
    )
    for i, (ax, (sym, lv, cv, ylabel)) in enumerate(zip(axes, METRICS)):
        if cv is None:
            ax.text(0.5, 0.5, "pending", transform=ax.transAxes,
                    ha="center", va="center", color=PENDING_COLOR, fontsize=13)
            ax.set_title(sym, fontsize=13, pad=6, fontweight="bold")
            continue
        _scatter_panel(ax, lv, cv, sym, sym, show_noise=(i == 0))
    _scatter_legend(fig)
    plt.tight_layout()
    fig.savefig(f"{OUT}/fig_pctc_scatter_1x4.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(f"{OUT}/fig_pctc_scatter_1x4.png", bbox_inches="tight", dpi=300)
    print("Saved fig_pctc_scatter_1x4")
    plt.close()


def fig_scatter_2x2():
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 9.5))
    fig.suptitle(
        r"PCTC $m=1$: Logical vs Compiled (H2-1E)" + "\nResource Overhead",
        fontsize=13, fontweight="bold", y=1.02
    )
    for i, (ax, (sym, lv, cv, ylabel)) in enumerate(zip(axes.flat, METRICS)):
        if cv is None:
            ax.text(0.5, 0.5, "pending", transform=ax.transAxes,
                    ha="center", va="center", color=PENDING_COLOR, fontsize=13)
            ax.set_title(sym, fontsize=13, pad=6, fontweight="bold")
            continue
        _scatter_panel(ax, lv, cv, sym, sym, show_noise=(i == 0))
    _scatter_legend(fig)
    plt.tight_layout()
    fig.savefig(f"{OUT}/fig_pctc_scatter_2x2.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(f"{OUT}/fig_pctc_scatter_2x2.png", bbox_inches="tight", dpi=300)
    print("Saved fig_pctc_scatter_2x2")
    plt.close()

In [16]:
fig_bar_1x4()
fig_bar_2x2()
fig_summary()
fig_scatter_1x4()
fig_scatter_2x2()
print("\nAll done.")
if not compiled_available:
    print("\nNOTE: Re-run after pctc_nexus_benchmark.py completes to get")
    print("      compiled metrics and fidelity estimate.")


Saved fig_pctc_bar_1x4
Saved fig_pctc_bar_2x2
Saved fig_pctc_summary
Saved fig_pctc_scatter_1x4
Saved fig_pctc_scatter_2x2

All done.
